# Notebook 00: Environment Setup and GEE Authentication

**Project:** Rural Settlement Detection and Population Undercount Quantification — East African Great Lakes Region  
**Run this notebook once before any other notebook.**

---

## Step 1: Create a Google Earth Engine Account

1. Go to [https://earthengine.google.com](https://earthengine.google.com)
2. Click **Get Started** and sign in with a Google account
3. Register for a **noncommercial / research** project (free for academic use)
4. Wait for approval email (usually same day for academic accounts)
5. Note your **GEE project ID** — you will need it below

---

## Step 2: Install Required Packages

Run the cell below. If you are on a fresh environment, this may take a few minutes.

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    'earthengine-api',
    'geemap',
    'numpy',
    'pandas',
    'geopandas',
    'rasterio',
    'scikit-learn',
    'matplotlib',
    'folium',
    'torch',
    'torchvision',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages installed successfully.')

## Step 3: Authenticate with Google Earth Engine

Run the cell below. It will open a browser window asking you to sign in and grant access.  
Copy the authorization code back into the prompt that appears in this notebook.

In [ ]:
import ee

# This opens a browser for OAuth authentication
# Follow the prompts, paste the code back here when asked
ee.Authenticate()

## Step 4: Set Your Project ID and Initialize GEE

In [ ]:
# -------------------------------------------------------
# EDIT THIS: replace with your actual GEE project ID
GEE_PROJECT_ID = 'your-gee-project-id-here'
# -------------------------------------------------------

ee.Initialize(project=GEE_PROJECT_ID)
print(f'GEE initialized successfully with project: {GEE_PROJECT_ID}')

## Step 5: Verify Connection

In [ ]:
# Quick connectivity test — should return a valid image info dict
test = ee.Image('COPERNICUS/S2_SR_HARMONIZED/20200101T075251_20200101T075251_T37MBN')
info = test.getInfo()
print('GEE connection verified.')
print(f'Test image ID: {info["id"]}')

## Step 6: Define Project-Wide Constants

These constants are imported by all subsequent notebooks.  
Do not change them after Stage 1 has been run, as downstream notebooks depend on these paths.

In [ ]:
import json, os

CONFIG = {
    # GEE project
    'gee_project': GEE_PROJECT_ID,

    # Study area — five-country Great Lakes extent
    # Bounding box: [west, south, east, north]
    'study_bbox': [28.85, -11.75, 41.90, 4.22],

    # Country list for iteration
    'countries': ['Kenya', 'Tanzania', 'Uganda', 'Rwanda', 'Burundi'],

    # GEE asset paths (GEE Cloud Storage export destinations)
    'gee_export_folder': 'east_africa_settlement',

    # Local output paths
    'data_dir': '../data',
    'output_dir': '../outputs',
    'model_dir': '../models',

    # Sentinel-2 bands used
    's2_bands': ['B02', 'B03', 'B04', 'B08'],

    # Sentinel-1 polarizations
    's1_bands': ['VV', 'VH'],

    # Composite resolution (metres)
    'composite_resolution_m': 10,

    # Confidence surface resolution (metres)
    'confidence_resolution_m': 500,

    # Confidence threshold for default filtered product
    'confidence_threshold': 0.4,

    # Isolation threshold (metres) for isolated vs clustered class assignment
    'isolation_threshold_m': 50,

    # GHS-SMOD class boundaries
    'smod_rural_classes': [10, 11, 12],
    'smod_periurban_classes': [21, 22],
    'smod_urban_classes': [23, 30],

    # Open Buildings confidence threshold for pseudo-labels
    'ob_confidence_threshold': 0.75,

    # Pseudo-label discount factor (d)
    'pseudo_label_discount': 0.5,

    # DHS displacement buffer (metres)
    'dhs_buffer_m': 5000,

    # Cloud cover filter for S2 scene selection
    's2_cloud_threshold_pct': 20,
}

# Save config to disk so all notebooks can load it
os.makedirs('../data', exist_ok=True)
with open('../data/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('Config saved to ../data/config.json')
print(json.dumps(CONFIG, indent=2))

## Step 7: Verify GHS-SMOD and Sentinel Collections Are Accessible

In [ ]:
import ee
ee.Initialize(project=GEE_PROJECT_ID)

# Check Sentinel-2 collection
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
print(f'S2 collection size (full archive): {s2.size().getInfo()} images')

# Check Sentinel-1 collection
s1 = ee.ImageCollection('COPERNICUS/S1_GRD')
print(f'S1 collection accessible: True')

# Check GHS-SMOD
smod = ee.Image('JRC/GHSL/P2023A/GHS_SMOD/2020')
print(f'GHS-SMOD 2020 accessible: True')

# Check ESA WorldCover
worldcover = ee.ImageCollection('ESA/WorldCover/v200')
print(f'ESA WorldCover accessible: True')

print('\nAll required GEE collections verified.')

---
## Setup Complete

You are ready to proceed to **Notebook 01: Sentinel Preprocessing**.

**Notebook execution order:**
1. `00_setup_auth.ipynb` ← you are here
2. `01_sentinel_preprocessing.ipynb` — cloud-free mosaic generation
3. `02_model_finetuning.ipynb` — Prithvi fine-tuning and inference
4. `03_crossvalidation.ipynb` — footprint product comparison
5. `04_confidence_population.ipynb` — confidence surface and bias quantification